In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf


df_raw = pd.read_csv("joined_glm.csv")


# Example structure of your cell-level modeling dataframe
# 'grazing_counts' = absolute number of foraging observations in that 10x10m cell
# 'total_observations' = total observations of ALL behaviours in that 10x10m cell
# 'habitat_type' = categorical variable (e.g., 'Grassland', 'Forest')
# 'dominant_cow_id' = the ID of the cow that occupied this cell the most (to control for individual)

df_raw.to_csv("data_for_glm_raw.csv", index=False)

df_raw

In [ ]:
profile = ( 
    df_raw.groupby(["NVC", "behaviour_state"]) #as_index=False
    .size() 
    .groupby(level=0) 
    .apply(lambda x: 100 * x / x.sum()) 
    #.reset_index(name="frequency")
) 

# 1. Wipe out the names from the multi-index completely to stop the name clash
profile.index.names = [None] * len(profile.index.names)

# 2. Reset the index safely now that the names are gone
df_clean1 = profile.reset_index(name="Allocation")

# 3. Explicitly overwrite the column names based on their positions
df_clean1.columns = ['Serial', 'Extra_Serial', 'behaviour_state', 'Allocation']

# 4. Drop the duplicate structural tracking column
df_clean1 = df_clean1.drop(columns=['Extra_Serial'])

df_clean1

In [ ]:
profile1 = ( 
    df_raw.groupby(["Serial", "behaviour_state"]) #as_index=False
    .size() 
    .groupby(level=0) 
    .apply(lambda x: 100 * x / x.sum()) 
    #.reset_index(name="frequency")
) 

# 1. Wipe out the names from the multi-index completely to stop the name clash
profile1.index.names = [None] * len(profile1.index.names)

# 2. Reset the index safely now that the names are gone
df_clean = profile1.reset_index(name="Allocation")

# 3. Explicitly overwrite the column names based on their positions
df_clean.columns = ['Serial', 'Extra_Serial', 'behaviour_state', 'Allocation']

# 4. Drop the duplicate structural tracking column
df_clean = df_clean.drop(columns=['Extra_Serial'])

df_clean

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 2. Set global publication typography
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.linewidth'] = 0.8

# 3. Pivot data and enforce a logical behavioral sequence
state_order = ["Resting", "Intensive Foraging", "Extensive Foraging", "Transit"]
df_pivot = df_clean1.pivot(index="Serial", columns="behaviour_state", values="Allocation")[state_order]

# 4. Map your exact journal-ready behavioral colors
custom_colors = {
    "Resting": "grey",            
    "Intensive Foraging": "lightgreen",   
    "Extensive Foraging": "darkgreen",   
    "Transit": "orange"                          
}

# 5. Initialize figure canvas (increase the vertical size if you have many animals)
fig, ax = plt.subplots(figsize=(6, 8), dpi=300)

# 6. Plot horizontal stacked bars (FIX: Keep width=0.6 even for barh)
df_pivot.plot(
    kind="barh", stacked=True, ax=ax, width=0.6,
    color=[custom_colors[s] for s in state_order],
    edgecolor="white", linewidth=0.5
)

# 7. Editorial formatting and axis cleanup
ax.set_title("Habitat Behaviour Time Budget Allocations", fontsize=11, fontweight="bold", pad=12, loc="left")
ax.set_ylabel("Habitat Classification", fontsize=10, labelpad=6)
ax.set_xlabel("Time Budget Allocation (%)", fontsize=10, labelpad=6)

ax.set_xlim(0, 100)  # Enforce strict percentage boundary limit
ax.set_yticklabels(df_pivot.index, fontsize=8)  
ax.tick_params(labelsize=9)

# Clean background grid lines (on the x-axis to track percentage segments)
ax.grid(axis='x', linestyle=':', alpha=0.4, color='#ced4da', zorder=0)
ax.set_axisbelow(True)

# Remove heavy top and right frame outlines (Minimalist despine)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

# 8. Position the legend box cleanly on the right hand side
ax.legend(
    loc="upper left", bbox_to_anchor=(1.02, 1.0),
    frameon=True, facecolor="white", edgecolor="#e9ecef", fontsize=9,
    title="Behaviour State", title_fontsize=9
)

plt.tight_layout()

# FIRST: Save the figure with a tight bounding box to prevent legend clipping
plt.savefig('habitat_budgets_b.png', dpi=300, bbox_inches='tight')

# SECOND: Show the figure on your screen
plt.show()

In [ ]:
gpipff = pd.read_csv("gpipff4.csv")

gpipff

In [ ]:
nvc_number = pd.read_csv("NVC_number.csv")

nvc_number

In [ ]:
# Assigns variables to an object.
continuous1 = ["Exp_p_day", "PFF"]

#Calculates descriptive statistics for numeric variables.
numeric1 = gpipff[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = gpipff[column].median()
print(numeric1)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Try importing adjustText to auto-resolve label overlaps. 
try:
    from adjustText import adjust_text
except ImportError:
    adjust_text = None

# 1. Set strict publication typography and styling
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['pdf.fonttype'] = 42  # Exports editable vector text for journals
plt.rcParams['ps.fonttype'] = 42

# Create figure matching double-column layout specifications
fig = plt.figure(figsize=(11, 5.5), dpi=300)

# Mock data initialization (Assumes gpipff is defined elsewhere in your environment)
habitat_summary = {
    'Habitat': gpipff["NVC"],
    'Expected Grazing Count Per Day': gpipff["Exp_p_day"],
    'PFF': gpipff["PFF"]
}
df_plot = pd.DataFrame(habitat_summary)

# Create the numeric IDs right after DataFrame initialization
df_plot['Habitat_ID'] = range(1, len(df_plot) + 1)

# Position the main plot axis explicitly to prevent collisions with the table
# format: [left, bottom, width, height]
ax = fig.add_axes([0.08, 0.12, 0.48, 0.75])

# 2. Draw quadrant background zones based on calculated thresholds
# Uses an ultra-subtle alpha (0.03) to ensure text readability
threshold_x =  4.330000
threshold_y = 0.059435

ax.axvspan(0.0, threshold_x, 0.11887, 1.0, color='#495057', alpha=0.03, zorder=0)  # Top Left
ax.axvspan(threshold_x, 12, 0.11887, 1.0, color='#c92a2a', alpha=0.03, zorder=0)  # Top Right

# 3. Draw threshold lines
ax.axvline(x=threshold_x, color='#6c757d', linestyle='--', linewidth=0.8, zorder=1)
ax.axhline(y=threshold_y, color='#6c757d', linestyle='--', linewidth=0.8, zorder=1)

# 4. Plot data points
sns.scatterplot(
    data=df_plot, x='Expected Grazing Count Per Day', y='PFF', 
    s=0, color='#005a9c', edgecolor='white', linewidth=0.6, zorder=4, ax=ax
)

# 5. Dynamic Quadrant Text (Aligned to axes fractions)
ax.text(0.03, 0.96, 'High Risk / Avoided', transform=ax.transAxes, 
        fontsize=8.5, color='#495057', fontweight='bold', va='top')
ax.text(0.97, 0.96, 'High Risk / Preferred\n(Vulnerable)', transform=ax.transAxes, 
        fontsize=8.5, color='#c92a2a', fontweight='bold', va='top', ha='right')
ax.text(0.03, 0.04, 'Low Risk / Avoided', transform=ax.transAxes, 
        fontsize=8.5, color='#6c757d', va='bottom')
ax.text(0.97, 0.04, 'Low Risk / Preferred', transform=ax.transAxes, 
        fontsize=8.5, color='#6c757d', va='bottom', ha='right')

# 6. Numeric labels and optimized adjust_text placement
texts = []
for i, row in df_plot.iterrows():
    t = ax.text(row['Expected Grazing Count Per Day'], row['PFF'], str(row['Habitat_ID']), 
                fontsize=7, fontweight='bold', va='center', ha='center', zorder=5)
    texts.append(t)

if adjust_text:
    adjust_text(
        texts, 
        arrowprops=dict(arrowstyle="-", color="#adb5bd", linewidth=0.5),
        force_points=(0.2, 0.4), 
        expand_points=(1.3, 1.3),
        ax=ax
    )

# 7. Axes limits and clean grid
ax.set_xlim(3.25, 5.25)
ax.set_ylim(0.0, 0.5)
ax.grid(True, linestyle=':', alpha=0.3, color='#ced4da', zorder=2)

# 8. Editorial labels and cleanup
ax.set_xlabel('Expected Grazing Count Per Day', fontsize=10, labelpad=6)
ax.set_ylabel('Proportional Foraging Footprint ($PFF_H$)', fontsize=10, labelpad=6)
ax.set_title('Habitat Spatial Vulnerability Matrix', fontsize=11, fontweight='bold', pad=12, loc='left')

# 9. Split and render the 33 items into a clean 2-column Legend Table
legend_data = list(zip(df_plot['Habitat_ID'], df_plot['Habitat']))
midpoint = (len(legend_data) + 1) // 2
col1_data = legend_data[:midpoint]
col2_data = legend_data[midpoint:]

table_rows = []
for i in range(midpoint):
    id1, name1 = col1_data[i]
    id2, name2 = col2_data[i] if i < len(col2_data) else ("", "")
    table_rows.append([id1, name1, id2, name2])

# Place table to the right side of the main plot area
table_ax = fig.add_axes([0.60, 0.12, 0.36, 0.75]) 
table_ax.axis('off')

matrix_table = table_ax.table(
    cellText=table_rows,
    colLabels=['No.', 'Habitat (NVC)', 'No.', 'Habitat (NVC)'],
    colWidths=[0.07, 0.43, 0.07, 0.43],
    loc='center',
    cellLoc='left'
)

# Apply publication-level font scaling and remove harsh interior lines
matrix_table.auto_set_font_size(False)
matrix_table.set_fontsize(7.5)
for (row_idx, col_idx), cell in matrix_table.get_celld().items():
    cell.set_linewidth(0.4)
    cell.set_edgecolor('#dee2e6')
    # Bold headers and index columns
    if row_idx == 0:
        cell.set_text_props(fontweight='bold')
        cell.set_facecolor('#f8f9fa')
    elif col_idx == 0 or col_idx == 2:
        cell.set_text_props(fontweight='bold')

sns.despine(ax=ax, trim=True) 

# Save outputs (Both high-res raster and infinite-resolution vector formats)
plt.savefig('Figure_2_Vulnerability_Matrix.png', bbox_inches='tight', dpi=300)
plt.savefig('Figure_2_Vulnerability_Matrix.pdf', bbox_inches='tight')
plt.show()